# 1. The Difference Between Funnel Analysis and Cohort Analysis


*   Funnel Analysis: Untuk menganalisis tahapan perjalanan customer dari awal (visit) hingga akhir (purchase atau churn).<br>**Tujuan:** Mengidentifikasi hambatan dan mengoptimalkan proses pengguna tertentu.

*   Cohort Analysis :  Untuk mengelompokkan pengguna berdasarkan karakteristik atau peristiwa yang sama yang terjadi pada saat yang bersamaan — seperti tanggal pembuatan akun atau investasi pertama — dan memeriksa perilaku mereka dari waktu ke waktu.<br>**Tujuan:** Memahami perilaku dan tren pengguna jangka panjang.



# 2. Creating a dataset for cohort purposes

## Data loading and innitial preparation

In [ ]:
# import library
import pandas as pd
import numpy as np

In [ ]:
# load your data
data_all_sheets = pd.read_excel('/content/Assignment Datasets Funnel Cohort.xlsx', sheet_name=['Events', 'Sales'])


In [ ]:
display(data_all_sheets['Events'].head())
display(data_all_sheets['Sales'].head())

,id,user_id,age,gender,city,country,session_id,sequence_number,created_at,browser,traffic_source,event_type
0,555311,42922.0,46.0,M,Bogatynia,Poland,f65f7f3f-1078-45da-a7af-2dd41a894e5d,3,2023-01-19 22:54:38 UTC,Other,Email,cart
1,714816,55343.0,16.0,M,Bogatynia,Poland,cc3509fc-641b-4632-aca7-4949209b9afb,4,2023-11-26 17:12:19 UTC,Safari,Email,cart
2,464405,35775.0,46.0,F,Bogatynia,Poland,16f8b065-9b2e-47bf-b8e7-793060cfbe04,9,2023-09-27 02:46:04 UTC,Firefox,Adwords,cart
3,649908,50243.0,47.0,M,Bogatynia,Poland,8b145bdf-4ddd-473a-8095-c35504afa112,3,2023-04-16 08:50:42 UTC,Chrome,Adwords,cart
4,874098,67437.0,30.0,M,Zgorzelec,Poland,4871ce1f-0b5c-446d-aa7c-016c959cbe56,3,2023-05-18 04:24:49 UTC,Firefox,Email,cart


,order_items_id,order_id,order_date,user_id,age,gender,product_id,product_name,product_category,product_price,product_cost,item_qty
0,7005,4790,2023-06-09,3802,18,F,8033,Embroidered Capri Set - Sizes: SMMEDLG,Clothing Sets,19.99,11.49,1
1,55182,38050,2023-03-17,30411,37,F,8038,Embroidered Capri Set - Sizes: 1X2X3X4X,Clothing Sets,22.99,13.86,1
2,107743,74517,2023-11-19,59668,28,F,8038,Embroidered Capri Set - Sizes: 1X2X3X4X,Clothing Sets,22.99,13.86,1
3,8139,5591,2023-05-13,4410,64,F,8028,Collections Etc - Autumn Floral V Neck Top And...,Clothing Sets,32.97,20.80,1
4,149787,103464,2023-05-04,82891,37,F,8028,Collections Etc - Autumn Floral V Neck Top And...,Clothing Sets,32.97,20.80,1


In [ ]:
# ensure data columns are in datetime format
data_all_sheets['Sales']['order_date'] = pd.to_datetime(data_all_sheets['Sales']['order_date'])
data_all_sheets['Events']['created_at'] = pd.to_datetime(data_all_sheets['Events']['created_at'])

In [ ]:
data_all_sheets['Sales'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37354 entries, 0 to 37353
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_items_id    37354 non-null  int64         
 1   order_id          37354 non-null  int64         
 2   order_date        37354 non-null  datetime64[ns]
 3   user_id           37354 non-null  int64         
 4   age               37354 non-null  int64         
 5   gender            37354 non-null  object        
 6   product_id        37354 non-null  int64         
 7   product_name      37351 non-null  object        
 8   product_category  37354 non-null  object        
 9   product_price     37354 non-null  float64       
 10  product_cost      37354 non-null  float64       
 11  item_qty          37354 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int64(6), object(3)
memory usage: 3.4+ MB


In [ ]:
data_all_sheets['Events'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418279 entries, 0 to 418278
Data columns (total 12 columns):
 #   Column           Non-Null Count   Dtype              
---  ------           --------------   -----              
 0   id               418279 non-null  int64              
 1   user_id          265687 non-null  float64            
 2   age              265687 non-null  float64            
 3   gender           265687 non-null  object             
 4   city             262839 non-null  object             
 5   country          265687 non-null  object             
 6   session_id       418279 non-null  object             
 7   sequence_number  418279 non-null  int64              
 8   created_at       418279 non-null  datetime64[ns, UTC]
 9   browser          418279 non-null  object             
 10  traffic_source   418279 non-null  object             
 11  event_type       418279 non-null  object             
dtypes: datetime64[ns, UTC](1), float64(2), int64(2), object(7)

## Determine the cohort group

In [ ]:
# find the firs transaction date for each customer
data_all_sheets['Sales']['CohortGroup'] = data_all_sheets['Sales'].groupby('user_id')['order_date'].transform('min')

# Extract the month and year for the cohort
data_all_sheets['Sales']['CohortGroup'] = data_all_sheets['Sales']['CohortGroup'].dt.to_period('M')


## Calculate Cohort Period (or Age):

In [ ]:
# Change the ‘CohortGroup’ column, which is of type Period, to type datetime with
data_all_sheets['Sales']['CohortGroup'] = data_all_sheets['Sales']['CohortGroup'].dt.to_timestamp()


In [ ]:
# the reduction between the two columns, which are now both of the datetime type
data_all_sheets['Sales']['CohortPeriod'] = (data_all_sheets['Sales']['order_date'] - data_all_sheets['Sales']['CohortGroup'])

In [ ]:
# Calculate the difference in months between the transaction and the cohort start
data_all_sheets['Sales']['CohortPeriod'] = ((data_all_sheets['Sales']['order_date'] - data_all_sheets['Sales']['CohortGroup']).dt.days / 30.4375).astype(int)

## Aggregate Data for Analysis:

In [ ]:
# # Calculate the number of unique customers in each cohort and period
cohort_counts = data_all_sheets['Sales'].groupby(['CohortGroup', 'CohortPeriod'])['user_id'].nunique().reset_index()

# rename the column for clarity
cohort_counts.rename(columns={'user_id': 'TotalUsers'}, inplace=True)

## Pivot for Cohort Analysis

In [ ]:
# create a pivot table for retention analysis
cohort_pivot = cohort_counts.pivot_table(index='CohortGroup', columns='CohortPeriod', values='TotalUsers')

# calculate retention rates
cohort_size = cohort_pivot.iloc[:, 0]
retention_matrix = cohort_pivot.divide(cohort_size, axis=0)

In [ ]:
# Ubah retention_matrix ke long format
retention_long = retention_matrix.reset_index().melt(
    id_vars=['CohortGroup'],      # kolom tetap
    var_name='CohortPeriod',      # kolom index (0,1,2,...)
    value_name='RetentionRate'    # nilai retention
)

# Simpan ke Excel
retention_long.to_excel("retention_long.xlsx", index=False)

# 3. Creating a Dataset for Funnel Purposes

In [ ]:
display(data_all_sheets['Events'].head())

,id,user_id,age,gender,city,country,session_id,sequence_number,created_at,browser,traffic_source,event_type
0,555311,42922.0,46.0,M,Bogatynia,Poland,f65f7f3f-1078-45da-a7af-2dd41a894e5d,3,2023-01-19 22:54:38+00:00,Other,Email,cart
1,714816,55343.0,16.0,M,Bogatynia,Poland,cc3509fc-641b-4632-aca7-4949209b9afb,4,2023-11-26 17:12:19+00:00,Safari,Email,cart
2,464405,35775.0,46.0,F,Bogatynia,Poland,16f8b065-9b2e-47bf-b8e7-793060cfbe04,9,2023-09-27 02:46:04+00:00,Firefox,Adwords,cart
3,649908,50243.0,47.0,M,Bogatynia,Poland,8b145bdf-4ddd-473a-8095-c35504afa112,3,2023-04-16 08:50:42+00:00,Chrome,Adwords,cart
4,874098,67437.0,30.0,M,Zgorzelec,Poland,4871ce1f-0b5c-446d-aa7c-016c959cbe56,3,2023-05-18 04:24:49+00:00,Firefox,Email,cart


In [ ]:
display(data_all_sheets['Sales'].head())

,order_items_id,order_id,order_date,user_id,age,gender,product_id,product_name,product_category,product_price,product_cost,item_qty,CohortGroup,CohortPeriod
0,7005,4790,2023-06-09,3802,18,F,8033,Embroidered Capri Set - Sizes: SMMEDLG,Clothing Sets,19.99,11.49,1,2023-06-01,0
1,55182,38050,2023-03-17,30411,37,F,8038,Embroidered Capri Set - Sizes: 1X2X3X4X,Clothing Sets,22.99,13.86,1,2023-03-01,0
2,107743,74517,2023-11-19,59668,28,F,8038,Embroidered Capri Set - Sizes: 1X2X3X4X,Clothing Sets,22.99,13.86,1,2023-11-01,0
3,8139,5591,2023-05-13,4410,64,F,8028,Collections Etc - Autumn Floral V Neck Top And...,Clothing Sets,32.97,20.80,1,2023-05-01,0
4,149787,103464,2023-05-04,82891,37,F,8028,Collections Etc - Autumn Floral V Neck Top And...,Clothing Sets,32.97,20.80,1,2023-05-01,0


In [ ]:
# sort by user and time to ensure correct event order
data_all_sheets['Events'].sort_values(by=['user_id', 'created_at'], inplace=True)
data_all_sheets['Sales'].sort_values(by=['user_id', 'order_date'], inplace=True)

## Create Funnel Progression Data

In [ ]:
data_events = data_all_sheets['Events'].copy()
data_sales = data_all_sheets['Sales'].copy()

display(data_events.head())
display(data_sales.head())

,id,user_id,age,gender,city,country,session_id,sequence_number,created_at,browser,traffic_source,event_type
211247,16,2.0,33.0,F,Ningbo,China,6fc28b78-e91a-4904-b0de-7054ddcc737c,1,2023-07-04 14:12:46+00:00,IE,YouTube,home
172261,18,2.0,33.0,F,Ningbo,China,6fc28b78-e91a-4904-b0de-7054ddcc737c,3,2023-07-04 14:15:52+00:00,IE,YouTube,product
155403,19,2.0,33.0,F,Ningbo,China,6fc28b78-e91a-4904-b0de-7054ddcc737c,4,2023-07-04 14:16:35+00:00,IE,YouTube,cart
228488,20,2.0,33.0,F,Ningbo,China,6fc28b78-e91a-4904-b0de-7054ddcc737c,5,2023-07-04 14:19:15+00:00,IE,YouTube,purchase
405602,21,3.0,37.0,F,Coventry,United Kingdom,03a5c11f-872b-48f5-8b42-e0d3fb34ee10,1,2023-07-30 16:28:04+00:00,IE,Email,home


,order_items_id,order_id,order_date,user_id,age,gender,product_id,product_name,product_category,product_price,product_cost,item_qty,CohortGroup,CohortPeriod
31569,4,4,2023-07-04,2,33,F,7706,GREEN KIMONO DUSTER JACKET SHORT STRETCH LYCRA...,Blazers & Jackets,50.99,21.72,1,2023-07-01,0
25787,25,20,2023-03-14,16,40,M,16218,U.S. Polo Assn. Men's Narrow Striped Polo With...,Tops & Tees,25.00,15.13,1,2023-03-01,0
789,47,38,2023-08-13,25,65,F,15882,Stormy Kromer Women's The Petal Pusher Cap,Plus,48.93,23.19,1,2023-08-01,0
36396,49,40,2023-09-16,26,40,M,17799,Enjoi Walk Among Us Sweater - Men's,Fashion Hoodies & Sweatshirts,54.99,28.32,1,2023-09-01,0
25788,58,45,2023-05-16,27,16,M,16012,Volcom Men's Font Various Short Sleeve Tee,Tops & Tees,25.00,13.55,1,2023-05-01,0


In [ ]:
data_events.sample(10)

,id,user_id,age,gender,city,country,session_id,sequence_number,created_at,browser,traffic_source,event_type
218739,914143,70484.0,26.0,M,Xianyang,China,054661c3-2c73-41a9-8c14-6f74c4b03493,5,2023-10-20 12:01:43+00:00,Chrome,Adwords,product
410514,639620,49468.0,59.0,M,Lowestoft,United Kingdom,6535280d-636c-4d1f-8288-8c2267556390,3,2023-08-29 02:44:20+00:00,Chrome,Adwords,cart
284224,495826,38230.0,55.0,F,São Félix do Xingu,Brasil,7e3bab58-ec66-45c0-818f-25f3963e9738,2,2023-07-26 00:31:22+00:00,Safari,Email,product
129479,1973075,NaN,NaN,NaN,NaN,NaN,2d2f326a-5a20-4749-9048-8c9a1fcdd631,2,2023-05-03 00:28:00+00:00,Chrome,Facebook,product
365647,427662,32993.0,33.0,M,Wayne,United States,e26aad79-9e30-4ef8-8f94-0117952efe32,2,2023-10-19 09:44:23+00:00,Firefox,Email,product
99297,1481316,NaN,NaN,NaN,NaN,NaN,324ed346-1d0e-444c-8cf5-4b51e6cf4520,2,2023-07-04 09:30:00+00:00,Chrome,Email,cart
255123,842839,65067.0,53.0,M,Sestao,Spain,45a2893b-3d7b-4bc9-a318-46777befc296,11,2023-06-15 09:07:05+00:00,Chrome,Email,product
2414,1886143,NaN,NaN,NaN,NaN,NaN,1de08d55-3d9a-438e-a234-6bb3b81d180f,3,2023-09-08 03:20:00+00:00,Chrome,Email,cancel
319132,637348,49338.0,38.0,M,Lübeck,Germany,c107a44d-9d39-4ef2-86d8-27671ee4bb12,12,2023-09-23 13:50:44+00:00,IE,Adwords,cart
387764,948589,73245.0,56.0,F,San Francisco,United States,6e456334-7348-4287-869f-cb152f3c5f06,3,2023-02-18 00:21:13+00:00,Other,Email,cart


In [ ]:
# Define the funnel stages
funnel_steps = ['home', 'product', 'cart', 'purchase', 'cancel']

# Filter the events data to include only the defined funnel stages
funnel_df = data_events[data_events['event_type'].isin(funnel_steps)].copy()

# Drop rows with missing user_id as they cannot be tracked through the funnel
funnel_df.dropna(subset=['user_id'], inplace=True)

# Display the first few rows of the funnel dataset
display(funnel_df.head())

,id,user_id,age,gender,city,country,session_id,sequence_number,created_at,browser,traffic_source,event_type
211247,16,2.0,33.0,F,Ningbo,China,6fc28b78-e91a-4904-b0de-7054ddcc737c,1,2023-07-04 14:12:46+00:00,IE,YouTube,home
172261,18,2.0,33.0,F,Ningbo,China,6fc28b78-e91a-4904-b0de-7054ddcc737c,3,2023-07-04 14:15:52+00:00,IE,YouTube,product
155403,19,2.0,33.0,F,Ningbo,China,6fc28b78-e91a-4904-b0de-7054ddcc737c,4,2023-07-04 14:16:35+00:00,IE,YouTube,cart
228488,20,2.0,33.0,F,Ningbo,China,6fc28b78-e91a-4904-b0de-7054ddcc737c,5,2023-07-04 14:19:15+00:00,IE,YouTube,purchase
405602,21,3.0,37.0,F,Coventry,United Kingdom,03a5c11f-872b-48f5-8b42-e0d3fb34ee10,1,2023-07-30 16:28:04+00:00,IE,Email,home


## Identify First Event for Each User at Each Stage

In [ ]:
# Sort by user and time to ensure correct event order
funnel_df.sort_values(by=['user_id', 'created_at'], inplace=True)

# Get the first event for each user at each stage
funnel_agg = funnel_df.groupby(['user_id', 'event_type'])['created_at'].min().reset_index()

# Pivot the table to have users as rows and funnel stages as columns
funnel_pivot = funnel_agg.pivot_table(index='user_id', columns='event_type', values='created_at')

# Display the first few rows of the pivoted funnel data
display(funnel_pivot.head())

event_type,cart,home,product,purchase
user_id,,,,
2.0,2023-07-04 14:16:35+00:00,2023-07-04 14:12:46+00:00,2023-07-04 14:15:52+00:00,2023-07-04 14:19:15+00:00
3.0,2023-07-30 16:33:14+00:00,2023-07-30 16:28:04+00:00,2023-07-30 16:31:28+00:00,2023-07-30 16:33:56+00:00
16.0,2023-03-14 16:41:31+00:00,2023-03-14 16:36:56+00:00,2023-03-14 16:40:02+00:00,2023-03-14 16:44:27+00:00
18.0,2023-03-09 21:21:06+00:00,NaT,2023-03-09 21:20:43+00:00,2023-03-12 23:07:18+00:00
25.0,2023-08-13 15:51:45+00:00,2023-08-13 15:44:24+00:00,2023-08-13 15:49:09+00:00,2023-08-13 15:54:12+00:00


## Calculate Funnel Metrics

In [ ]:
# Calculate the number of users at each stage
funnel_counts = funnel_pivot.count()

# Calculate the number of users who progressed to the next stage
funnel_conversions = funnel_pivot.notna().sum()

# Create a DataFrame to display the funnel metrics based on available steps
funnel_metrics = pd.DataFrame({
    'Step': funnel_conversions.index,
    'UserCount': funnel_conversions.values
})

# Add the 'cancel' step with the total count of cancel events
cancel_row = pd.DataFrame({'Step': ['cancel'], 'UserCount': [cancel_count_total]})
funnel_metrics = pd.concat([funnel_metrics, cancel_row], ignore_index=True)

# Ensure the steps are in the desired order for the plot
funnel_metrics['Step'] = pd.Categorical(funnel_metrics['Step'], categories=funnel_steps, ordered=True)
funnel_metrics.sort_values('Step', inplace=True)

# Recalculate conversion rates based on the updated UserCount column
funnel_metrics['ConversionRate'] = funnel_metrics['UserCount'].pct_change().fillna(1.0)

display(funnel_metrics)

,Step,UserCount,ConversionRate
1,home,21296,1.000000
2,product,28921,0.358048
0,cart,28921,0.000000
3,purchase,28941,0.000692
4,cancel,21768,-0.247849


In [ ]:
# Save the funnel metrics to an Excel file
funnel_metrics.to_excel('funnel_metrics.xlsx', index=False)

In [ ]:
import plotly.express as px

# Create a funnel plot
fig = px.funnel(funnel_metrics, x='UserCount', y='Step', title='Funnel Analysis')
fig.show()